<a href="https://colab.research.google.com/github/rifaa05/Project-Akhir_ML_5C_Kelompok-2/blob/main/02_praproses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Praproses Data

**Proyek Akhir Pembelajaran Mesin (INF62325) — Kelas 5C**  
**Paket:** 5C-3 Heart Failure Prediction  
**Anggota:** Adel Zamalya (2488010059), Adam Fadli (2488010028), Rifa’ah Fatihatu Sa’adah (2488010008)

Notebook ini menerapkan keputusan dari EDA dan menghasilkan `data/data_bersih.csv`. Pembersihan data mentah dipisahkan dari transformasi untuk model. Secara khusus, imputasi median, penskalaan, dan one-hot encoding **tidak** dilakukan sebelum pembagian latih-uji agar tidak terjadi data leakage.

## 1. Memuat data mentah dan memeriksa ukuran

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/heart.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/heart.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Letakkan file sumber sebagai data/heart.csv sebelum menjalankan notebook.")

OUTPUT_PATH = Path("../data/data_bersih.csv")
if not OUTPUT_PATH.parent.exists():
    OUTPUT_PATH = Path("data/data_bersih.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
ukuran_awal = df.shape
print("Ukuran awal:", ukuran_awal)
print("Missing eksplisit awal:", int(df.isna().sum().sum()))
print("Duplikat awal:", int(df.duplicated().sum()))
display(df.head())

Ukuran awal: (918, 12)
Missing eksplisit awal: 0
Duplikat awal: 0


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## 2. Validasi nilai dan kategori

Pastikan target hanya berisi 0 dan 1, variabel biner sesuai domain, serta nilai nol yang menjadi perhatian memang ada.

In [ ]:
required_columns = ["Age", "Sex", "ChestPainType", "RestingBP", "Cholesterol", "FastingBS", "RestingECG", "MaxHR", "ExerciseAngina", "Oldpeak", "ST_Slope", "HeartDisease"]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

assert set(df["HeartDisease"].dropna().unique()).issubset({0, 1}), "Target harus berupa 0/1"
assert set(df["FastingBS"].dropna().unique()).issubset({0, 1}), "FastingBS harus berupa 0/1"

print("RestingBP=0:", int((df["RestingBP"] == 0).sum()))
print("Cholesterol=0:", int((df["Cholesterol"] == 0).sum()))
print("Duplikat:", int(df.duplicated().sum()))

RestingBP=0: 1
Cholesterol=0: 172
Duplikat: 0


## 3. Penanganan missing value tersamar

Menurut catatan khusus paket 5C-3, `Cholesterol=0` pada sekitar 170 baris dan `RestingBP=0` pada satu baris merupakan nilai hilang tersamar. Kedua nilai nol tersebut diubah menjadi `NaN`, bukan diganti dengan nol atau dihapus otomatis.

**Keputusan:** mempertahankan seluruh baris dan menandai nilai sebagai missing. Kolom `Cholesterol` memiliki proporsi missing cukup besar (sekitar 18,7%). Menghapus semua baris yang terkena akan mengurangi jumlah observasi secara berarti dan berpotensi mengubah komposisi kelas. Nilai median akan dipelajari dari data latih saja pada pipeline pemodelan.

In [ ]:
df_bersih = df.copy()

for col in ["RestingBP", "Cholesterol"]:
    df_bersih.loc[df_bersih[col] == 0, col] = np.nan

print("Jumlah missing setelah penandaan:")
display(pd.DataFrame({
    "jumlah": df_bersih.isna().sum(),
    "persen": (df_bersih.isna().mean() * 100).round(2)
}))
print("Ukuran setelah penandaan:", df_bersih.shape)

Jumlah missing setelah penandaan:


,jumlah,persen
Age,0,0.00
Sex,0,0.00
ChestPainType,0,0.00
RestingBP,1,0.11
Cholesterol,172,18.74
FastingBS,0,0.00
RestingECG,0,0.00
MaxHR,0,0.00
ExerciseAngina,0,0.00
Oldpeak,0,0.00


Ukuran setelah penandaan: (918, 12)


## 4. Pemeriksaan duplikat, kolom konstan, dan kategori

Tidak menghapus pencilan IQR secara otomatis. Nilai ekstrem tetap dipertahankan kecuali terbukti mustahil secara domain. Duplikat identik diperiksa ulang sebelum menyimpan.

In [ ]:
duplikat = int(df_bersih.duplicated().sum())
print("Duplikat identik sebelum keputusan:", duplikat)
if duplikat > 0:
    # Hanya hapus duplikat identik karena tidak ada identifier kejadian yang membedakan baris.
    df_bersih = df_bersih.drop_duplicates().copy()
print("Ukuran setelah pemeriksaan duplikat:", df_bersih.shape)

constant_cols = df_bersih.nunique(dropna=False)
constant_cols = constant_cols[constant_cols <= 1].index.tolist()
print("Kolom konstan:", constant_cols)
if constant_cols:
    df_bersih = df_bersih.drop(columns=constant_cols)

cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
for col in cat_cols:
    df_bersih[col] = df_bersih[col].astype("string").str.strip()
    print(col, sorted(df_bersih[col].dropna().unique().tolist()))

print("Ukuran akhir setelah pemeriksaan:", df_bersih.shape)

Duplikat identik sebelum keputusan: 0
Ukuran setelah pemeriksaan duplikat: (918, 12)
Kolom konstan: []
Sex ['F', 'M']
ChestPainType ['ASY', 'ATA', 'NAP', 'TA']
RestingECG ['LVH', 'Normal', 'ST']
ExerciseAngina ['N', 'Y']
ST_Slope ['Down', 'Flat', 'Up']
Ukuran akhir setelah pemeriksaan: (918, 12)


## 5. Tabel keputusan praproses

Tabel berikut merupakan jejak keputusan yang dapat diperiksa kembali. Baris yang terdampak dihitung dari data mentah.

In [ ]:
decision_table = pd.DataFrame([
    ["RestingBP", "1 nilai 0", "Ubah 0 menjadi NaN", "Nol tidak mungkin sebagai tekanan darah istirahat; median diimputasi hanya dari data latih pada pipeline", "1 sel"],
    ["Cholesterol", "172 nilai 0", "Ubah 0 menjadi NaN", "Nol tidak mungkin sebagai kolesterol serum; pertahankan baris karena penghapusan 172 observasi dapat menimbulkan kehilangan informasi", "172 sel"],
    ["Baris duplikat", f"{int(df.duplicated().sum())} baris identik", "Hapus hanya jika ditemukan", "Baris identik tidak menambah informasi; pada dataset ini tidak ditemukan", f"{int(df.duplicated().sum())} baris"],
    ["Kolom konstan", "Tidak ditemukan", "Tidak ada penghapusan", "Tidak ada kolom dengan satu nilai unik", "0 kolom"],
    ["Pencilan IQR", "Kandidat pencilan pada beberapa fitur", "Pertahankan", "Pencilan statistik belum tentu nilai mustahil; tidak ada dasar domain untuk menghapus otomatis", "Tidak ada baris dihapus"],
    ["Kategori", "Kategori diperiksa dan spasi tepi dibersihkan", "Strip spasi; pertahankan sebagai kategori", "Mencegah label yang sama terbaca berbeda akibat spasi", "Kolom kategorikal"]
], columns=["Kolom/objek", "Temuan", "Keputusan", "Alasan", "Baris/sel terpengaruh"])
display(decision_table)

,Kolom/objek,Temuan,Keputusan,Alasan,Baris/sel terpengaruh
0,RestingBP,1 nilai 0,Ubah 0 menjadi NaN,Nol tidak mungkin sebagai tekanan darah istira...,1 sel
1,Cholesterol,172 nilai 0,Ubah 0 menjadi NaN,Nol tidak mungkin sebagai kolesterol serum; pe...,172 sel
2,Baris duplikat,0 baris identik,Hapus hanya jika ditemukan,Baris identik tidak menambah informasi; pada d...,0 baris
3,Kolom konstan,Tidak ditemukan,Tidak ada penghapusan,Tidak ada kolom dengan satu nilai unik,0 kolom
4,Pencilan IQR,Kandidat pencilan pada beberapa fitur,Pertahankan,Pencilan statistik belum tentu nilai mustahil;...,Tidak ada baris dihapus
5,Kategori,Kategori diperiksa dan spasi tepi dibersihkan,Strip spasi; pertahankan sebagai kategori,Mencegah label yang sama terbaca berbeda akiba...,Kolom kategorikal


## 6. Simpan dataset bersih

`data_bersih.csv` berisi data dengan nilai nol yang tidak valid sudah ditandai sebagai nilai kosong (`NaN`). File ini **belum** diimputasi atau diskalakan. Langkah tersebut dilakukan dalam pipeline setelah train-test split.

In [ ]:
df_bersih.to_csv(OUTPUT_PATH, index=False)
print("Ukuran awal :", ukuran_awal)
print("Ukuran bersih:", df_bersih.shape)
print("File tersimpan:", OUTPUT_PATH.resolve())
print("Missing tersisa:")
display(df_bersih.isna().sum().to_frame("jumlah"))
print("Preview data bersih:")
display(df_bersih.head())

Ukuran awal : (918, 12)
Ukuran bersih: (918, 12)
File tersimpan: /mnt/data/ProyekAkhir_5C_HeartFailure/data/data_bersih.csv
Missing tersisa:


,jumlah
Age,0
Sex,0
ChestPainType,0
RestingBP,1
Cholesterol,172
FastingBS,0
RestingECG,0
MaxHR,0
ExerciseAngina,0
Oldpeak,0


Preview data bersih:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140.0,289.0,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160.0,180.0,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130.0,283.0,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138.0,214.0,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150.0,195.0,0,Normal,122,N,0.0,Up,0


## 7. Ketentuan untuk tahap pemodelan

Saat membuat model klasifikasi, lakukan pembagian data terlebih dahulu (`train_test_split`). Setelah itu, gunakan `SimpleImputer(strategy="median")` untuk numerik dan `SimpleImputer(strategy="most_frequent")` untuk kategorikal di dalam `ColumnTransformer`/`Pipeline`. Untuk k-NN, lanjutkan dengan `StandardScaler` pada fitur numerik dan `OneHotEncoder(handle_unknown="ignore")` pada fitur kategorikal. Seluruh transformer di-fit hanya pada data latih.

Contoh kerangka (belum melatih model):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df_bersih.drop(columns="HeartDisease")
y = df_bersih["HeartDisease"]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_knn = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Data latih:", X_train.shape)
print("Data uji:", X_test.shape)
print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)
print("Preprocessor siap dipasang pada Pipeline model k-NN.")

Data latih: (734, 11)
Data uji: (184, 11)
Fitur numerik: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Fitur kategorikal: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
Preprocessor siap dipasang pada Pipeline model k-NN.


## 8. Kesimpulan praproses

Dataset memiliki 918 baris dan 12 kolom. Nilai nol pada `RestingBP` dan `Cholesterol` ditandai sebagai missing tersamar. Baris dipertahankan agar kehilangan observasi tidak terlalu besar. Data bersih disimpan tanpa imputasi global; imputasi dan penskalaan dilakukan di pipeline setelah pemisahan data untuk menghindari kebocoran informasi dari data uji.